In [1]:
# 1) Project setup and imports
from pathlib import Path
import torch
import torch.nn as nn

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "scripts" / "train.py").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

print("CUDA available:", torch.cuda.is_available())

CUDA available: True


In [2]:
# 2) Import reusable training components from scripts/train.py (with reload)
import sys
import importlib

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import scripts.train as train_mod
importlib.reload(train_mod)

set_seed = train_mod.set_seed
build_dataloaders = train_mod.build_dataloaders
AthenAIModel = train_mod.AthenAIModel
COMMAND_VOCAB = train_mod.COMMAND_VOCAB
TARGET_NUM_SAMPLES = train_mod.TARGET_NUM_SAMPLES
NUM_SENSORS = train_mod.NUM_SENSORS

set_seed(42)
print("Reloaded scripts.train and loaded", len(COMMAND_VOCAB), "command classes")

/home/nithira/Multimodal_Intent_Reconstruction_for_Safety_Critical_Communication_in_Extreme_Acoustic_Environments/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Reloaded scripts.train and loaded 20 command classes


In [3]:
# 3) Build dataloaders
BATCH_SIZE = 16
DATA_DIR = PROJECT_ROOT / "data" / "mixed"

dataloaders = build_dataloaders(data_dir=DATA_DIR, batch_size=BATCH_SIZE)
train_loader = dataloaders["train"]
val_loader = dataloaders["val"]
test_loader = dataloaders["test"]

print("Train samples:", len(train_loader.dataset))
print("Val samples:", len(val_loader.dataset))
print("Test samples:", len(test_loader.dataset))

Train samples: 35267
Val samples: 4400
Test samples: 4429


In [ ]:
# 4) Build model + optimizer + loss
# Architecture: WavJEPA → Intent Query Decoder (IQD)
# IQD replaces mean-pool+FC with 20 learned class queries that cross-attend
# over the full WavJEPA temporal sequence. Ablation: +21.6% vs FC baseline.
MODE    = "base"   # audio-only with IQD
DECODER = "iqd"    # Intent Query Decoder
device  = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = AthenAIModel(mode=MODE, decoder_type=DECODER).to(device)
trainable_params = [p for p in model.parameters() if p.requires_grad]

optimizer = torch.optim.AdamW(trainable_params, lr=3e-4, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss(label_smoothing=0.05)  # label smoothing improves ECE

print("Mode:", MODE, "| Decoder:", DECODER)
print("Device:", device)
print("Trainable params:", sum(p.numel() for p in trainable_params))
print("Frozen params:",   sum(p.numel() for p in model.parameters() if not p.requires_grad))

In [5]:
# 5) One-batch sanity check
audio, sensor, target = next(iter(train_loader))

print("Audio shape:", tuple(audio.shape), "expected [B, 32000]")
print("Sensor shape:", tuple(sensor.shape), "expected [B, 128, 8]")
print("Target shape:", tuple(target.shape))

audio = audio.to(device)
sensor = sensor.to(device)
target = target.to(device)

model.train()
if MODE == "full":
    logits = model(audio, sensor)
else:
    logits = model(audio)

loss = criterion(logits, target)
print("Logits shape:", tuple(logits.shape), "expected [B, 20]")
print("Sanity loss:", float(loss.item()))

Audio shape: (16, 32000) expected [B, 32000]
Sensor shape: (16, 128, 8) expected [B, 128, 8]
Target shape: (16,)


/home/nithira/Multimodal_Intent_Reconstruction_for_Safety_Critical_Communication_in_Extreme_Acoustic_Environments/.venv/lib/python3.10/site-packages/torch/masked/maskedtensor/creation.py:19: UserWarning: The PyTorch API of MaskedTensors is in prototype stage and will change in the near future. Please open a Github issue for features requests and see our documentation on the torch.masked module for further information about the project.
  return MaskedTensor(data, mask, requires_grad)


Logits shape: (16, 20) expected [B, 20]
Sanity loss: 3.0903897285461426


In [ ]:
# 6) Start training — IQD, 30 epochs, base mode (audio-only)
# Saves best checkpoint to: checkpoints/best_iqd.pt
import subprocess

cmd = [
    ".venv/bin/python", "scripts/train.py",
    "--mode",    MODE,
    "--decoder", DECODER,
    "--epochs",  "30",
    "--batch_size", str(BATCH_SIZE),
    "--lr",       "3e-4",
]

print("Running:", " ".join(cmd))
subprocess.run(cmd, cwd=PROJECT_ROOT, check=True)

In [ ]:
# 7) Run evaluation after training completes
# Evaluates the IQD checkpoint (checkpoints/best_iqd.pt)
# Outputs: eval_results.json, eval_reliability.png, eval_uncertainty.png
eval_cmd = [
    ".venv/bin/python", "scripts/evaluate.py",
    "--mode",       MODE,
    "--checkpoint", "checkpoints/best_iqd.pt",
]

print("Running:", " ".join(eval_cmd))
subprocess.run(eval_cmd, cwd=PROJECT_ROOT, check=True)

In [ ]:
# 8) Quick usage guide
print("Run cells 1-5 for setup and sanity checks.")
print("In cell 6, uncomment subprocess.run(...) to start training.")
print("After training, in cell 7 uncomment subprocess.run(...) to run evaluation.")
print("Evaluation outputs: eval_results.json, eval_reliability.png, eval_uncertainty.png")